# SchemeSaathi Bench: LoRA fine-tuning (arms 3 and 4)

Self-contained. Run top to bottom on a **free Colab T4** (Runtime > Change runtime type > T4 GPU),
or in VS Code with the Colab extension.

What it does:
1. Clones the repo and reads `data/sft/train.jsonl` and `data/sft/val.jsonl`
   (made by `python -m saathi.finetune.export_sft`; commit them to your branch first).
2. LoRA-fine-tunes the base model with Unsloth + TRL, loss on assistant turns only.
3. Converts the fine-tuned model (merged 16-bit) **and the untouched base model** to GGUF
   (F16, Q8_0, Q4_K_M) with one llama.cpp checkout, so arms 1-2 and 3-4 differ only in weights.
4. Writes `run_info.json`: settings, data hashes, losses, package versions.

The install cell follows Unsloth's official `Qwen3_(4B)-Instruct` notebook (checked 2026-10-23).

In [ ]:
# ---- settings (everything that affects the result is here) ----
REPO_URL = "https://github.com/naqi7272/SchemeSaathi"
BRANCH = "main"
BASE_MODEL = "unsloth/Qwen3-1.7B"
OUT_NAME = "qwen3-1.7b-saathi"
MAX_SEQ_LEN = 3072  # RAG prompts with 5 passages; Hindi text uses more tokens
LORA_R, LORA_ALPHA = 16, 32
LEARNING_RATE = 2e-4
EPOCHS = 2
BATCH, GRAD_ACCUM = 2, 4  # effective batch 8
SEED = 3407
BASE_HF = "Qwen/Qwen3-1.7B"  # original weights, converted for arms 1-2
QUANTS = ["Q8_0", "Q4_K_M"]  # plus F16

In [ ]:
%%capture
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

In [ ]:
import hashlib
import json
import os
import subprocess

subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "repo"], check=True)
DATA = {"train": "repo/data/sft/train.jsonl", "val": "repo/data/sft/val.jsonl"}
DATA_SHA256 = {k: hashlib.sha256(open(v, "rb").read()).hexdigest() for k, v in DATA.items()}
print({k: sum(1 for _ in open(v, encoding="utf-8")) for k, v in DATA.items()}, DATA_SHA256)

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files=DATA)


def to_text(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
            for m in batch["messages"]
        ]
    }


ds = ds.map(to_text, batched=True)
lengths = [len(tokenizer(t).input_ids) for t in ds["train"]["text"]]
print("max tokens:", max(lengths), "| over limit:", sum(n > MAX_SEQ_LEN for n in lengths))

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=10,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=SEED,
        output_dir="outputs",
        report_to="none",
    ),
)
# loss only on the assistant's answer, not on the prompt
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
final_eval = trainer.evaluate()
print(stats.metrics, final_eval)

In [ ]:
# Sanity check: answer three validation prompts (greedy).
FastLanguageModel.for_inference(model)
for row in ds["val"].select(range(3)):
    prompt = tokenizer.apply_chat_template(
        row["messages"][:-1], tokenize=False, add_generation_prompt=True
    )
    ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(**ids, max_new_tokens=200, do_sample=False)
    print("GOLD:", row["messages"][-1]["content"])
    print("GOT: ", tokenizer.decode(out[0][ids.input_ids.shape[1] :], skip_special_tokens=True))
    print("-" * 60)

In [ ]:
model.save_pretrained(f"{OUT_NAME}-lora")
tokenizer.save_pretrained(f"{OUT_NAME}-lora")
model.save_pretrained_merged(f"{OUT_NAME}-merged", tokenizer, save_method="merged_16bit")

In [ ]:
# One llama.cpp checkout converts both models (about 10 minutes to build).
from huggingface_hub import snapshot_download

!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF > /dev/null
!cmake --build llama.cpp/build --config Release -j --target llama-quantize > /dev/null
LLAMA_CPP_COMMIT = subprocess.run(
    ["git", "-C", "llama.cpp", "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip()

base_dir = snapshot_download(BASE_HF)

os.makedirs("gguf", exist_ok=True)
for name, src in [("qwen3-1.7b-base", base_dir), (OUT_NAME, f"{OUT_NAME}-merged")]:
    f16 = f"gguf/{name}.F16.gguf"
    subprocess.run(
        ["python", "llama.cpp/convert_hf_to_gguf.py", src, "--outfile", f16, "--outtype", "f16"],
        check=True,
    )
    for q in QUANTS:
        subprocess.run(
            ["llama.cpp/build/bin/llama-quantize", f16, f"gguf/{name}.{q}.gguf", q], check=True
        )
!ls -lh gguf

In [ ]:
import importlib.metadata as md

info = {
    "base_model": BASE_MODEL,
    "max_seq_len": MAX_SEQ_LEN,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "batch": BATCH,
    "grad_accum": GRAD_ACCUM,
    "seed": SEED,
    "quants": QUANTS,
    "base_hf": BASE_HF,
    "llama_cpp_commit": LLAMA_CPP_COMMIT,
    "data_sha256": DATA_SHA256,
    "train_metrics": stats.metrics,
    "final_eval": final_eval,
    "log_history": trainer.state.log_history,
    "versions": {
        p: md.version(p)
        for p in ["unsloth", "unsloth_zoo", "trl", "peft", "transformers", "torch", "datasets"]
    },
}
json.dump(info, open("run_info.json", "w"), indent=2, default=str)

## Get the files to your laptop

GGUF files are large (4B: roughly 2.5 GB at Q4_K_M, 4.3 GB at Q8_0, 8 GB at F16; two models).
Pick one option:

- **Google Drive** (run the next cell), then download the `gguf` folder from Drive.
- **Hugging Face Hub** (private repo): uncomment the last cell; set `HF_TOKEN` in Colab secrets.

Then on the laptop: `uv run python -m saathi.deploy.ollama_models gguf/`
creates the Ollama models used by the eval configs. Also copy `run_info.json` to
`results/finetune/run_info.json` in the repo and commit it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/schemesaathi
!cp -r gguf run_info.json {OUT_NAME}-lora /content/drive/MyDrive/schemesaathi/

In [ ]:
# from google.colab import userdata
# from huggingface_hub import HfApi
# HfApi(token=userdata.get("HF_TOKEN")).upload_folder(
#     folder_path="gguf", repo_id="YOUR_HF_USER/schemesaathi-gguf", repo_type="model")